In [2]:
import chromadb
from langchain_groq import ChatGroq
import yfinance as yf
import pandas as pd

In [ ]:
# Setup Groq
llm = ChatGroq(
    api_key="Replace With Yours",
    model="meta-llama/llama-4-scout-17b-16e-instruct"
)

# Setup ChromaDB
client = chromadb.Client()
collection = client.create_collection("finance_knowledge")

print("ChromaDB and Groq ready!")

ChromaDB and Groq ready!


In [4]:
def load_stock_into_knowledge_base(ticker):
    stock = yf.Ticker(ticker)
    info = stock.info
    news = stock.news
    
    # Prepare documents
    documents = []
    ids = []
    
    # Add company info as a document
    company_doc = f"""
    Company: {info.get('longName', 'N/A')}
    Sector: {info.get('sector', 'N/A')}
    Industry: {info.get('industry', 'N/A')}
    Market Cap: {info.get('marketCap', 'N/A')}
    PE Ratio: {info.get('trailingPE', 'N/A')}
    52W High: {info.get('fiftyTwoWeekHigh', 'N/A')}
    52W Low: {info.get('fiftyTwoWeekLow', 'N/A')}
    Current Price: {info.get('currentPrice', 'N/A')}
    """
    documents.append(company_doc)
    ids.append(f"{ticker}_info")
    
    # Add each news article as a document
    for i, article in enumerate(news[:5]):
        title = article.get("content", {}).get("title", "N/A")
        summary = article.get("content", {}).get("summary", "N/A")
        news_doc = f"News: {title}. Summary: {summary}"
        documents.append(news_doc)
        ids.append(f"{ticker}_news_{i}")
    
    # Store in ChromaDB
    collection.add(documents=documents, ids=ids)
    print(f"✅ {ticker} loaded into knowledge base! ({len(documents)} documents)")

print("Knowledge base loader ready!")

Knowledge base loader ready!


In [5]:
load_stock_into_knowledge_base("AAPL")
load_stock_into_knowledge_base("TSLA")
load_stock_into_knowledge_base("GOOGL")

✅ AAPL loaded into knowledge base! (6 documents)
✅ TSLA loaded into knowledge base! (6 documents)
✅ GOOGL loaded into knowledge base! (6 documents)


In [6]:
def ask_knowledge_base(question, n_results=3):
    # Search ChromaDB for relevant documents
    results = collection.query(
        query_texts=[question],
        n_results=n_results
    )
    
    # Combine retrieved documents
    context = "\n\n".join(results["documents"][0])
    
    # Build prompt with context
    prompt = f"""
    You are an expert financial analyst with access to a knowledge base.
    Use the following context to answer the question accurately.
    
    Context:
    {context}
    
    Question: {question}
    
    Provide a clear and concise answer based only on the context provided.
    """
    
    response = llm.invoke(prompt)
    return response.content

print("RAG query function ready!")

RAG query function ready!


In [7]:
answer = ask_knowledge_base("What is Apple's current price and PE ratio?")
print(answer)

Apple's current price is $271.35 and its PE ratio is 34.348103.


In [8]:
# Indian stocks need .NS suffix for NSE
tickers = ["BAGFILMS.NS", "ADANIPOWER.NS"]

for ticker in tickers:
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        print(f"\n--- {ticker} ---")
        print(f"Company: {info.get('longName', 'N/A')}")
        print(f"Current Price: {info.get('currentPrice', 'N/A')}")
        print(f"Market Cap: {info.get('marketCap', 'N/A')}")
    except Exception as e:
        print(f"❌ {ticker} Error: {e}")


--- BAGFILMS.NS ---
Company: B.A.G. Films and Media Limited
Current Price: 5.22
Market Cap: 1084288384

--- ADANIPOWER.NS ---
Company: Adani Power Limited
Current Price: 222.08
Market Cap: 4282745094144


In [9]:
load_stock_into_knowledge_base("BAGFILMS.NS")
load_stock_into_knowledge_base("ADANIPOWER.NS")

✅ BAGFILMS.NS loaded into knowledge base! (1 documents)
✅ ADANIPOWER.NS loaded into knowledge base! (6 documents)


In [10]:
questions = [
    "What is Adani Power's current price and market cap?",
    "Tell me about BAG Films and Media?",
    "Which is more valuable, Adani Power or BAG Films?"
]

for question in questions:
    print(f"\nQ: {question}")
    print(f"A: {ask_knowledge_base(question)}")
    print("-" * 50)


Q: What is Adani Power's current price and market cap?
A: Adani Power's current price is ₹222.08, and its market capitalization (Market Cap) is ₹4,282,745,094,144.
--------------------------------------------------

Q: Tell me about BAG Films and Media?
A: B.A.G. Films and Media Limited is a company in the Broadcasting industry, classified under the Communication Services sector. Here are some key highlights:

* Market Capitalization: ₹1084288384
* Current Price: ₹5.22
* 52-Week High/Low: ₹8.0 / ₹3.61
* Price-to-Earnings (PE) Ratio: 23.727272

The stock is currently trading at ₹5.22, which is below its 52-week high of ₹8.0 but above its 52-week low of ₹3.61. The PE ratio suggests that the stock may be reasonably valued compared to its earnings. However, a more comprehensive analysis would require additional information and data.
--------------------------------------------------

Q: Which is more valuable, Adani Power or BAG Films?
A: Based on the context provided, it's not possible t